<a href="https://colab.research.google.com/github/lucasarabi/DiceMan/blob/main/First_Algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import numpy as np

df_NVDA = pd.read_csv('/content/data/HistoricalData_1759930571734.csv')

# --- Config ---
QUANTILE = 0.70            # trigger on largest 30% intraday moves
ROUND_TRIP_COST_BPS = 10.0 # open+close cost in bps
LONG_ONLY = False          # if True, shorts become flat (0)

# --- Helpers ---
def find_col(df, candidates):
    # exact match first
    for c in candidates:
        for col in df.columns:
            if col.strip().lower() == c.lower():
                return col
    # substring match next
    for c in candidates:
        for col in df.columns:
            if c.lower() in col.strip().lower():
                return col
    return None

def to_num(series):
    # robust numeric parsing: strips $, commas, % etc.
    return pd.to_numeric(series.astype(str).str.replace(r"[^0-9.\-eE]", "", regex=True), errors="coerce")

def normalize_ohlcv_from_df(df):
    DATE_CANDS  = ["date", "time", "timestamp"]
    OPEN_CANDS  = ["open", "o", "opening price"]
    HIGH_CANDS  = ["high", "h", "max", "high price"]
    LOW_CANDS   = ["low", "l", "min", "low price"]
    CLOSE_CANDS = ["close", "c", "adj close", "last", "closing price", "price"]
    VOL_CANDS   = ["volume", "vol", "trades"]

    date_col = find_col(df, DATE_CANDS)
    if date_col is None:
        # fallback: first parseable column
        for col in df.columns:
            try:
                pd.to_datetime(df[col])
                date_col = col
                break
            except Exception:
                pass
    if date_col is None:
        raise ValueError("No date/timestamp column found.")

    out = pd.DataFrame()
    out["date"] = pd.to_datetime(df[date_col], errors="coerce", utc=False, infer_datetime_format=True)
    out = out.dropna(subset=["date"]).copy()

    for name, cands in [("open", OPEN_CANDS), ("high", HIGH_CANDS),
                        ("low", LOW_CANDS), ("close", CLOSE_CANDS)]:
        col = find_col(df, cands)
        if col is None:
            raise ValueError(f"Missing required column for {name}.")
        out[name] = to_num(df.loc[out.index, col])

    vol_col = find_col(df, VOL_CANDS)
    out["volume"] = to_num(df.loc[out.index, vol_col]) if vol_col else np.nan

    out = out.dropna(subset=["open", "high", "low", "close"]).sort_values("date").reset_index(drop=True)
    return out

def backtest_gap_range_fade(df_ohlc, quantile=0.0, rt_cost_bps=10.0, long_only=False):
    """
    Build signal from today's open->close return (oc_ret).
    Big up -> short tomorrow; big down -> long tomorrow.
    Trades are entered at next day's open and exited same day's close (1-day hold).
    """
    oc_ret = df_ohlc["close"] / df_ohlc["open"] - 1.0
    thr = oc_ret.abs().quantile(quantile)

    sig = pd.Series(0.0, index=df_ohlc.index)
    sig[oc_ret >=  thr] = -1.0
    sig[oc_ret <= -thr] = +1.0
    if long_only:
        sig = sig.where(sig > 0, 0.0)

    # Position today = signal from yesterday
    pos = np.sign(sig).shift(1).fillna(0.0)

    # One-day P&L from open->close on the same day
    strat_ret_gross = pos * oc_ret

    # Apply fixed round-trip cost when we take a position
    cost = (rt_cost_bps / 10000.0) * (pos.abs() > 0).astype(float)
    strat_ret = strat_ret_gross - cost

    equity = (1.0 + strat_ret).cumprod()
    ann_factor = 252.0
    ann_ret = equity.iloc[-1]**(ann_factor/len(equity)) - 1.0 if len(equity) else np.nan
    ann_vol = strat_ret.std(ddof=0) * np.sqrt(ann_factor)
    sharpe = ann_ret / ann_vol if (ann_vol and ann_vol != 0) else np.nan
    hit_rate = (strat_ret > 0).mean()
    max_dd = (equity / equity.cummax() - 1.0).min() if len(equity) else np.nan

    return {
        "threshold": float(thr),
        "position": pos,
        "daily_returns": strat_ret,
        "equity": equity,
        "metrics": {
            "traded_days": int((pos.abs() > 0).sum()),
            "ann_return": float(ann_ret) if pd.notna(ann_ret) else np.nan,
            "ann_vol": float(ann_vol) if pd.notna(ann_vol) else np.nan,
            "sharpe": float(sharpe) if pd.notna(sharpe) else np.nan,
            "hit_rate": float(hit_rate) if pd.notna(hit_rate) else np.nan,
            "max_drawdown": float(max_dd) if pd.notna(max_dd) else np.nan,
        },
    }

df_ohlc = normalize_ohlcv_from_df(df_NVDA)

res = backtest_gap_range_fade(
    df_ohlc,
    quantile=QUANTILE,
    rt_cost_bps=ROUND_TRIP_COST_BPS,
    long_only=LONG_ONLY
)

print("=== Gap/Range Fade ===")
print(f"Quantile: {QUANTILE:.2f}  |  |oc_ret| >= {res['threshold']:.6g}")
if LONG_ONLY:
    print("Mode: LONG-ONLY (no shorts)")
print("Metrics:")
for k, v in res["metrics"].items():
    print(f"- {k}: {v}")

=== Gap/Range Fade ===
Quantile: 0.70  |  |oc_ret| >= 0.023927
Metrics:
- traded_days: 377
- ann_return: 0.19858815097075033
- ann_vol: 0.26114422877629817
- sharpe: 0.7604539143036749
- hit_rate: 0.16334661354581673
- max_drawdown: -0.28296624923163716


/tmp/ipython-input-1623819761.py:51: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["date"] = pd.to_datetime(df[date_col], errors="coerce", utc=False, infer_datetime_format=True)


## Parameter & cost grid (does the edge survive small tweaks?)

In [ ]:
import itertools

quantiles = [0.60, 0.70, 0.80, 0.90]
costs     = [5, 10, 20, 30, 50]
modes     = [False, True]  # LONG_ONLY False/True

rows = []
for q, c, lo in itertools.product(quantiles, costs, modes):
    res = backtest_gap_range_fade(df_ohlc, quantile=q, rt_cost_bps=c, long_only=lo)
    m = res["metrics"]
    rows.append({
        "quantile": q, "cost_bps": c, "long_only": lo,
        "ann_return": m["ann_return"], "ann_vol": m["ann_vol"],
        "sharpe": m["sharpe"], "hit_rate": m["hit_rate"],
        "traded_days": m["traded_days"], "max_dd": m["max_drawdown"]
    })

grid = pd.DataFrame(rows).sort_values(["long_only","quantile","cost_bps"])
print(grid.to_string(index=False))

 quantile  cost_bps  long_only  ann_return  ann_vol    sharpe  hit_rate  traded_days    max_dd
      0.6         5      False    0.289488 0.296955  0.974855  0.223904          502 -0.304441
      0.6        10      False    0.226223 0.296679  0.762515  0.219920          502 -0.343126
      0.6        20      False    0.108767 0.296280  0.367109  0.215139          502 -0.421771
      0.6        30      False    0.002461 0.296085  0.008313  0.205578          502 -0.508783
      0.6        50      False   -0.180799 0.296307 -0.610174  0.190438          502 -0.754019
      0.7         5      False    0.244721 0.261457  0.935988  0.167331          377 -0.260260
      0.7        10      False    0.198588 0.261144  0.760454  0.163347          377 -0.282966
      0.7        20      False    0.111326 0.260670  0.427077  0.160159          377 -0.350340
      0.7        30      False    0.030339 0.260397  0.116511  0.154582          377 -0.429197
      0.7        50      False   -0.114561 0.26046

## Regime tests (does it only work in one period?)

In [ ]:
def run_period(df, start=None, end=None, **kwargs):
    d = df.copy()
    if start: d = d[d["date"] >= pd.to_datetime(start)]
    if end:   d = d[d["date"] <= pd.to_datetime(end)]
    if len(d) < 60:
        return {"period": f"{start or '-inf'}→{end or '+inf'}", "note": "too few days"}
    res = backtest_gap_range_fade(d, **kwargs)
    m = res["metrics"]
    return {
        "period": f"{start or '-inf'}→{end or '+inf'}",
        "ann_return": m["ann_return"], "ann_vol": m["ann_vol"],
        "sharpe": m["sharpe"], "hit_rate": m["hit_rate"],
        "traded_days": m["traded_days"], "max_dd": m["max_drawdown"]
    }

periods = [
    ("2019-01-01","2021-12-31"),
    ("2022-01-01","2023-12-31"),
    ("2024-01-01","2025-12-31"),
]

rows = [run_period(df_ohlc, s, e, quantile=QUANTILE, rt_cost_bps=ROUND_TRIP_COST_BPS, long_only=LONG_ONLY)
        for (s, e) in periods]
print(pd.DataFrame(rows).to_string(index=False))


               period  ann_return  ann_vol    sharpe  hit_rate  traded_days    max_dd
2019-01-01→2021-12-31    0.171636 0.235958  0.727399  0.176849           94 -0.157450
2022-01-01→2023-12-31   -0.009189 0.266854 -0.034433  0.147705          151 -0.333808
2024-01-01→2025-12-31    0.543298 0.264865  2.051225  0.176072          133 -0.114061


## LABEL AFTER

In [ ]:
for c in [5, 10, 15, 20, 30, 50, 75, 100]:
    m = backtest_gap_range_fade(df_ohlc, quantile=QUANTILE, rt_cost_bps=c, long_only=LONG_ONLY)["metrics"]
    print(f"cost={c:>3} bps | Sharpe={m['sharpe']:.2f}  AnnRet={m['ann_return']:.2%}  MaxDD={m['max_drawdown']:.2%}")

cost=  5 bps | Sharpe=0.94  AnnRet=24.47%  MaxDD=-26.03%
cost= 10 bps | Sharpe=0.76  AnnRet=19.86%  MaxDD=-28.30%
cost= 15 bps | Sharpe=0.59  AnnRet=15.41%  MaxDD=-30.72%
cost= 20 bps | Sharpe=0.43  AnnRet=11.13%  MaxDD=-35.03%
cost= 30 bps | Sharpe=0.12  AnnRet=3.03%  MaxDD=-42.92%
cost= 50 bps | Sharpe=-0.44  AnnRet=-11.46%  MaxDD=-63.70%
cost= 75 bps | Sharpe=-1.02  AnnRet=-26.77%  MaxDD=-84.47%
cost=100 bps | Sharpe=-1.49  AnnRet=-39.46%  MaxDD=-93.43%
